# 🧠 NLP Sentiment Analysis — IMDB Movie Reviews

**Author:** Uche Jeremiah Nzubechukwu  
**Stack:** Python · Pandas · Scikit-learn · NLTK · TF-IDF · Matplotlib  
**Data:** [IMDB Dataset of 50K Movie Reviews](https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews) — Kaggle  

## Overview
An end-to-end NLP pipeline that classifies movie reviews as **Positive** or **Negative** using the industry-standard IMDB dataset of 50,000 real reviews. This project demonstrates text preprocessing, TF-IDF feature extraction, multi-model comparison, and a live inference function ready for API deployment.

## Pipeline
1. Load & explore real IMDB review data
2. Text preprocessing (HTML cleaning, stopwords, stemming)
3. TF-IDF feature extraction with bigrams
4. Train & compare 3 models: Naive Bayes, Logistic Regression, Linear SVM
5. Evaluation: Accuracy, F1, ROC-AUC, Confusion Matrix
6. Error analysis — what did the model get wrong?
7. Live `predict_sentiment()` inference function

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import re
from collections import Counter

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, ConfusionMatrixDisplay,
                              roc_auc_score, roc_curve)
from sklearn.calibration import CalibratedClassifierCV
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
print('✅ Libraries loaded')

## 1. Load Real IMDB Dataset
50,000 movie reviews from IMDB — perfectly balanced: 25,000 positive, 25,000 negative.

In [ ]:
DATASET_PATH = 'IMDB Dataset.csv'

try:
    df = pd.read_csv(DATASET_PATH)
    print(f'✅ Dataset loaded: {DATASET_PATH}')
except FileNotFoundError:
    print(f'❌ File not found: {DATASET_PATH}')
    print('   Download from: https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews')
    raise

print(f'\n📊 Dataset Info:')
print(f'   Shape      : {df.shape[0]:,} reviews × {df.shape[1]} columns')
print(f'   Columns    : {list(df.columns)}')
print(f'   Pos/Neg    : {df["sentiment"].value_counts().to_dict()}')
print(f'   Avg length : {df["review"].str.len().mean():.0f} characters')
print(f'   Max length : {df["review"].str.len().max():,} characters')
df.head(3)

## 2. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('IMDB Sentiment Dataset — Exploratory Data Analysis', fontsize=14, fontweight='bold')

# Class balance
counts = df['sentiment'].value_counts()
axes[0,0].bar(counts.index, counts.values,
              color=['#27AE60', '#E74C3C'], edgecolor='white', linewidth=2, width=0.5)
for i, v in enumerate(counts.values):
    axes[0,0].text(i, v + 200, f'{v:,}\n({v/len(df):.0%})',
                   ha='center', fontweight='bold', fontsize=11)
axes[0,0].set_title('Class Distribution', fontweight='bold')
axes[0,0].set_ylabel('Count')
axes[0,0].set_ylim(0, counts.max() * 1.15)

# Review length distribution
df['review_len'] = df['review'].str.len()
df['word_count'] = df['review'].str.split().str.len()
for sentiment, color in [('positive','#27AE60'), ('negative','#E74C3C')]:
    subset = df[df['sentiment'] == sentiment]['review_len']
    axes[0,1].hist(subset, bins=50, alpha=0.6, color=color, label=sentiment, density=True)
axes[0,1].set_title('Review Length Distribution', fontweight='bold')
axes[0,1].set_xlabel('Character Count')
axes[0,1].set_ylabel('Density')
axes[0,1].set_xlim(0, 5000)
axes[0,1].legend()

# Word count distribution
for sentiment, color in [('positive','#27AE60'), ('negative','#E74C3C')]:
    subset = df[df['sentiment'] == sentiment]['word_count']
    axes[1,0].hist(subset, bins=50, alpha=0.6, color=color, label=sentiment, density=True)
axes[1,0].set_title('Word Count Distribution', fontweight='bold')
axes[1,0].set_xlabel('Word Count')
axes[1,0].set_ylabel('Density')
axes[1,0].set_xlim(0, 800)
axes[1,0].legend()
pos_avg = df[df['sentiment']=='positive']['word_count'].mean()
neg_avg = df[df['sentiment']=='negative']['word_count'].mean()
axes[1,0].axvline(pos_avg, color='#27AE60', linestyle='--', lw=2, label=f'Pos avg: {pos_avg:.0f}w')
axes[1,0].axvline(neg_avg, color='#E74C3C', linestyle='--', lw=2, label=f'Neg avg: {neg_avg:.0f}w')
axes[1,0].legend(fontsize=9)

# Top words in positive reviews (after basic cleaning)
pos_text = ' '.join(df[df['sentiment']=='positive']['review'].str.lower().values)
pos_text = re.sub(r'[^a-z\s]', ' ', pos_text)
stopwords_basic = {'the','a','an','and','or','but','in','on','at','to','for','of','with',
                   'is','was','it','this','that','i','he','she','they','we','my','his',
                   'her','its','are','be','been','have','has','had','do','not','as','by',
                   'from','so','if','than','very','just','also','no','more','all','were',
                   'one','would','there','their','what','about','which','up','out','who',
                   'br','film','movie','movies','films','even','good','great','really'}
pos_words = [w for w in pos_text.split() if w not in stopwords_basic and len(w) > 3]
top_pos = Counter(pos_words).most_common(12)
words, freqs = zip(*top_pos)
axes[1,1].barh(words, freqs, color='#27AE60', alpha=0.8, edgecolor='white')
axes[1,1].set_title('Top Words — Positive Reviews', fontweight='bold')
axes[1,1].set_xlabel('Frequency')
axes[1,1].invert_yaxis()

plt.tight_layout()
plt.savefig('imdb_eda.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ EDA chart saved: imdb_eda.png')

## 3. Text Preprocessing
IMDB reviews contain HTML tags, special characters, and noise that must be cleaned before modelling.

In [ ]:
STOPWORDS = {
    'i','me','my','myself','we','our','you','your','he','him','his','she','her',
    'it','its','they','them','their','what','which','who','this','that','these',
    'those','am','is','are','was','were','be','been','being','have','has','had',
    'do','does','did','a','an','the','and','but','if','or','as','at','by','for',
    'with','of','to','in','on','up','so','then','than','very','just','br','also'
}

def simple_stem(word):
    for suffix in ['ing', 'tion', 'ness', 'ful', 'less', 'ly', 'ed', 'er', 'est', 'ive']:
        if word.endswith(suffix) and len(word) > len(suffix) + 3:
            return word[:-len(suffix)]
    return word

def preprocess(text):
    text = re.sub(r'<[^>]+>', ' ', text)           # Remove HTML tags
    text = re.sub(r'https?://\S+', ' ', text)       # Remove URLs
    text = re.sub(r"[^a-zA-Z']", ' ', text)         # Keep only letters
    text = text.lower()
    tokens = text.split()
    tokens = [simple_stem(w) for w in tokens
              if w not in STOPWORDS and len(w) > 2]
    return ' '.join(tokens)

print('🔧 Preprocessing 50,000 reviews...')
df['clean_review'] = df['review'].apply(preprocess)
df['label']        = (df['sentiment'] == 'positive').astype(int)

print('✅ Preprocessing complete!')
print(f'\nSample:')
print(f'  Raw    : {df["review"].iloc[0][:120]}...')
print(f'  Cleaned: {df["clean_review"].iloc[0][:120]}...')

## 4. Train/Test Split
Using a stratified 80/20 split to preserve the 50/50 class balance.

In [ ]:
# Use 20k sample for faster training — remove .sample() to use full 50k
df_sample = df.sample(n=20000, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(
    df_sample['clean_review'], df_sample['label'],
    test_size=0.20, random_state=42, stratify=df_sample['label']
)

print(f'📊 Split Summary:')
print(f'   Total sample : {len(df_sample):,} reviews')
print(f'   Train        : {len(X_train):,} | Pos: {y_train.sum():,} | Neg: {(y_train==0).sum():,}')
print(f'   Test         : {len(X_test):,}  | Pos: {y_test.sum():,}  | Neg: {(y_test==0).sum():,}')
print(f'\n💡 Tip: Remove .sample(n=20000) to train on full 50k reviews')

## 5. Model Training & Comparison
Three classical NLP models with TF-IDF bigram features. Each wrapped in a `Pipeline` for clean, reproducible code.

In [ ]:
models = {
    'Naive Bayes': Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1,2), max_features=50000,
                                  sublinear_tf=True, min_df=2)),
        ('clf',   MultinomialNB(alpha=0.1))
    ]),
    'Logistic Regression': Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1,2), max_features=50000,
                                  sublinear_tf=True, min_df=2)),
        ('clf',   LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs'))
    ]),
    'Linear SVM': Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1,2), max_features=50000,
                                  sublinear_tf=True, min_df=2)),
        ('clf',   CalibratedClassifierCV(LinearSVC(C=1.0, max_iter=2000)))
    ]),
}

results = {}
print('🔧 Training models on real IMDB reviews...\n')

for name, pipeline in models.items():
    pipeline.fit(X_train, y_train)
    y_pred  = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]
    acc  = accuracy_score(y_test, y_pred)
    auc  = roc_auc_score(y_test, y_proba)
    cv   = cross_val_score(pipeline, df_sample['clean_review'],
                            df_sample['label'],
                            cv=StratifiedKFold(5), scoring='accuracy').mean()
    results[name] = {'acc': acc, 'auc': auc, 'cv': cv,
                     'y_pred': y_pred, 'y_proba': y_proba, 'model': pipeline}
    print(f'  {name:<22} → Acc: {acc:.4f}  AUC: {auc:.4f}  CV-Acc: {cv:.4f}')

best_name  = max(results, key=lambda k: results[k]['auc'])
best       = results[best_name]
print(f'\n🏆 Best Model: {best_name} (AUC={best["auc"]:.4f})')

## 6. Evaluation & Visualizations

In [ ]:
print(f'\n📄 Classification Report — {best_name}')
print('=' * 55)
print(classification_report(y_test, best['y_pred'],
                             target_names=['Negative', 'Positive']))

fig, axes = plt.subplots(2, 2, figsize=(14, 11))
fig.suptitle(f'IMDB Sentiment Analysis — Model Results', fontsize=14, fontweight='bold')

# Confusion Matrix
cm = confusion_matrix(y_test, best['y_pred'])
ConfusionMatrixDisplay(cm, display_labels=['Negative', 'Positive']).plot(
    ax=axes[0,0], colorbar=False, cmap='Blues')
axes[0,0].set_title(f'Confusion Matrix — {best_name}', fontweight='bold')

# ROC Curves
colors_roc = ['#3498DB', '#27AE60', '#E74C3C']
for (name, res), color in zip(results.items(), colors_roc):
    fpr, tpr, _ = roc_curve(y_test, res['y_proba'])
    axes[0,1].plot(fpr, tpr, lw=2, color=color,
                   label=f'{name} (AUC={res["auc"]:.4f})')
axes[0,1].plot([0,1],[0,1],'k--',lw=1,label='Random')
axes[0,1].set_title('ROC Curves — All Models', fontweight='bold')
axes[0,1].set_xlabel('False Positive Rate')
axes[0,1].set_ylabel('True Positive Rate')
axes[0,1].legend(fontsize=9)

# Model comparison
names_ = list(results.keys())
x = np.arange(len(names_))
axes[1,0].bar(x - 0.2, [results[n]['acc'] for n in names_], 0.35,
              label='Test Accuracy', color='#2E5FA3', alpha=0.85)
axes[1,0].bar(x + 0.2, [results[n]['cv'] for n in names_],  0.35,
              label='CV Accuracy',   color='#27AE60', alpha=0.85)
axes[1,0].set_xticks(x)
axes[1,0].set_xticklabels(names_, rotation=10, fontsize=9)
axes[1,0].set_ylim(0.7, 1.0)
axes[1,0].set_title('Model Comparison', fontweight='bold')
axes[1,0].set_ylabel('Accuracy')
axes[1,0].legend(fontsize=9)
for i, n in enumerate(names_):
    axes[1,0].text(i-0.2, results[n]['acc']+0.003, f"{results[n]['acc']:.3f}",
                   ha='center', fontsize=8, fontweight='bold')

# Top predictive features (Logistic Regression coefficients)
lr_model  = results['Logistic Regression']['model']
tfidf_vec = lr_model.named_steps['tfidf']
lr_clf    = lr_model.named_steps['clf']
feature_names = tfidf_vec.get_feature_names_out()
coefs         = lr_clf.coef_[0]
top_pos_idx   = coefs.argsort()[-10:][::-1]
top_neg_idx   = coefs.argsort()[:10]
top_features  = ([feature_names[i] for i in top_neg_idx] +
                  [feature_names[i] for i in top_pos_idx])
top_coefs     = ([coefs[i] for i in top_neg_idx] +
                  [coefs[i] for i in top_pos_idx])
bar_colors    = ['#E74C3C' if c < 0 else '#27AE60' for c in top_coefs]
y_pos = range(len(top_features))
axes[1,1].barh(y_pos, top_coefs, color=bar_colors, alpha=0.85, edgecolor='white')
axes[1,1].set_yticks(y_pos)
axes[1,1].set_yticklabels(top_features, fontsize=9)
axes[1,1].axvline(0, color='black', lw=1)
axes[1,1].set_title('Top Predictive Words\n(Green=Positive, Red=Negative)',
                     fontweight='bold')
axes[1,1].set_xlabel('Coefficient (LR)')

plt.tight_layout()
plt.savefig('imdb_model_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Results chart saved: imdb_model_results.png')

## 7. Error Analysis — What Did the Model Get Wrong?
Understanding failures is just as important as measuring accuracy.

In [ ]:
test_df = df_sample.iloc[y_test.index].copy()
test_df['predicted'] = best['y_pred']
test_df['correct']   = (test_df['label'] == test_df['predicted'])
test_df['confidence']= best['y_proba']

errors = test_df[~test_df['correct']].copy()
print(f'🔍 Error Analysis — {best_name}')
print(f'   Total errors   : {len(errors)} / {len(test_df)} ({len(errors)/len(test_df):.1%})')

# False Positives: predicted positive but actually negative
fp = errors[errors['predicted'] == 1].sort_values('confidence', ascending=False)
# False Negatives: predicted negative but actually positive
fn = errors[errors['predicted'] == 0].sort_values('confidence')

print(f'\n   False Positives (predicted POS, actually NEG): {len(fp)}')
print(f'   False Negatives (predicted NEG, actually POS): {len(fn)}')

print(f'\n📌 Sample False Positives (model was confidently wrong):')
for _, row in fp.head(2).iterrows():
    review_preview = row['review'][:200].replace('\n', ' ')
    print(f'   Confidence: {row["confidence"]:.2%}')
    print(f'   Review    : "{review_preview}..."')
    print()

print(f'📌 Sample False Negatives (model missed positive sentiment):')
for _, row in fn.head(2).iterrows():
    review_preview = row['review'][:200].replace('\n', ' ')
    print(f'   Confidence: {row["confidence"]:.2%}')
    print(f'   Review    : "{review_preview}..."')
    print()

print('💡 Insight: Errors often occur on sarcastic, mixed, or nuanced reviews')
print('   → Solution: Fine-tune BERT which understands context, not just word frequency')

## 8. Live Inference — Try It Yourself

In [ ]:
def predict_sentiment(text: str, model=best['model']) -> dict:
    """
    Predict sentiment of any movie review.

    Args:
        text: raw review string (HTML and noise handled automatically)
    Returns:
        dict with sentiment label, confidence, and cleaned text
    """
    clean   = preprocess(text)
    label   = model.predict([clean])[0]
    proba   = model.predict_proba([clean])[0]
    conf    = proba.max()
    sentiment = 'POSITIVE 😊' if label == 1 else 'NEGATIVE 😠'
    strength  = 'Strong' if conf > 0.85 else 'Moderate' if conf > 0.65 else 'Weak'
    return {'sentiment': sentiment, 'confidence': f'{conf:.1%}', 'strength': strength}


test_reviews = [
    "This is one of the greatest films I have ever seen. The acting was superb and the story kept me hooked throughout.",
    "Absolutely terrible. A complete waste of two hours. The plot made no sense and the acting was wooden and unconvincing.",
    "It started off really well but fell apart in the second half. Some good performances though.",
    "I can't believe how bad this movie was. The director clearly had no idea what they were doing.",
    "A masterpiece of modern cinema. Every scene is beautifully crafted. Will definitely watch again.",
    "Interesting concept but poor execution. The visual effects were impressive but the story was lacking.",
]

print(f'🤖 Live Sentiment Predictions — {best_name}')
print('=' * 65)
for review in test_reviews:
    result = predict_sentiment(review)
    preview = review[:70] + '...' if len(review) > 70 else review
    print(f'\n  Review     : "{preview}"')
    print(f'  Sentiment  : {result["sentiment"]}')
    print(f'  Confidence : {result["confidence"]} ({result["strength"]})')

## 9. Production Notes & Upgrade Path

### What This Demonstrates
- End-to-end NLP pipeline on a **real, industry-standard dataset**
- Proper text preprocessing handling real-world noise (HTML tags, special characters)
- Model comparison with statistical validation (stratified CV)
- Error analysis — understanding model failures, not just successes
- Deployment-ready `predict_sentiment()` function

### Upgrade Path
```
1. Fine-tune BERT/RoBERTa for ~93%+ accuracy (handles sarcasm & context)
2. Add aspect-based sentiment (not just positive/negative, but WHY)
3. Deploy as FastAPI endpoint: POST /predict → JSON response
4. Extend to multi-class: 1–5 star rating prediction
5. Real-time Twitter/Reddit sentiment dashboard
```

### Production API (FastAPI)
```python
from fastapi import FastAPI
app = FastAPI()

@app.post('/predict')
def predict(payload: dict):
    result = predict_sentiment(payload['review'])
    return result
```